# Feature Engineering  

This notebook uses the cleaned data from the folder data/processed and creates additional features.  

In [78]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from ydata_profiling import ProfileReport
from c08_farming_exit import config, features, data_cleaning, mappings, feature_engineering

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [79]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Import processed data

In [80]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "clean_data.csv")

## 2. Employment features

### 2.1 Employment categories

In [81]:
#EMPLOYMENT CATEGORIES
conditions = [
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].isnull()),
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].notnull()),
    (df["farm_empl_last_12_months"] == 0) & (df["empl_type"].notnull()),
]
choices = ["only_farm", "hybrid", "fully_off_farm"]

df["empl_category"] = np.select(conditions, choices, default=None)

### 2.2 Work hours per year - absolute numbers

In [82]:
#FARMING
df["cash_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_cash_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_cash_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_cash_crops_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_cash_crops_hours_per_day"],
    np.nan
)

df["food_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_food_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_food_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_food_crops_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_food_crops_hours_per_day"],
    np.nan
)

# No filtering here: livestock owners that don't do crop farming might not claim that they have worked on the farm. 
df["livestock_hours_per_year"] = (
    (df["farm_empl_livestock_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_livestock_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_livestock_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_livestock_hours_per_day"]
)

df["farm_empl_hours_per_year"] = df[["cash_crop_hours_per_year", "food_crop_hours_per_year", "livestock_hours_per_year"]].sum(axis=1, min_count=1)

In [83]:
#SELF-EMPLOYMENT
df["self_empl_hours_per_year"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_duration_rainy_season_in_months_last_12_months"]
     + df["self_empl_duration_dry_season_in_months_last_12_months"])
    * (df["self_empl_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["self_empl_hours_per_day"],
    np.nan
)

In [84]:
#PERMANENT WAGE EMPLOYMENT
df["wage_empl_permanent_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Permanent",
    12
    * (df["wage_empl_permanent_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["wage_empl_permanent_hours_per_day"],
    np.nan
)

In [85]:
#SEASONAL/CAUSAL WAGE EMPLOYMENT
df["wage_empl_seasonal_casual_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Seasonal",
    (df["wage_empl_seasonal_casual_rainy_season_duration_in_months_last_12_months"]
     + df["wage_empl_seasonal_casual_dry_season_duration_in_months_last_12_months"])
    * (df["wage_empl_seasonal_casual_duration_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["wage_empl_seasonal_casual_duration_hours_per_day"],
    np.nan
)

In [86]:
#TOTAL YEARLY WORK HOURS
cols = [
    "farm_empl_hours_per_year",
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]
df["total_work_hours_per_year"] = df[cols].sum(axis=1, min_count=1)

### 2.3 Work hours per year - relative numbers

In [87]:
#WORK TYPE SHARES
# avoid dividing by zero -> treat a total of 0 hours as NaN (undefined share)
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)

df["farm_hours_annual_share"] = df["farm_empl_hours_per_year"] / total_safe
df["self_empl_hours_annual_share"] = df["self_empl_hours_per_year"] / total_safe
df["wage_empl_permanent_hours_annual_share"] = df["wage_empl_permanent_hours_per_year"] / total_safe
df["wage_empl_seasonal_casual_hours_annual_share"] = df["wage_empl_seasonal_casual_hours_per_year"] / total_safe

In [88]:
# OFF-FARM WORK SHARE
off_farm_cols = [
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]

# sum off-farm categories, treating "not applicable" (NaN) as 0 
df["off_farm_hours_per_year"] = df[off_farm_cols].sum(axis=1, min_count=0)

# but if the person has NO work data at all, keep it NaN rather than 0
df.loc[df["total_work_hours_per_year"].isna(), "off_farm_hours_per_year"] = np.nan

# share of total work time that is off-farm
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)
df["off_farm_hours_annual_share"] = df["off_farm_hours_per_year"] / total_safe
df["on_farm_hours_annual_share"] = 1 - df["off_farm_hours_annual_share"]

### 2.4 Hourly and annual wage - absolute numbers

In [89]:
#FARMING
country_wage_map = {
    "Botswana": feature_engineering.agricultural_wage_per_hour(df, "Botswana",    payment_frequency="Per month",  agriculture_only=False) ,
    "Kenya":    feature_engineering.agricultural_wage_per_hour(df, "Kenya",       payment_frequency="Per day",    agriculture_only=True) ,
    "Namibia":  feature_engineering.agricultural_wage_per_hour(df, "Namibia",     payment_frequency="Per month",  agriculture_only=False) ,
    "Tanzania": feature_engineering.agricultural_wage_per_hour(df, "Tanzania",    payment_frequency="Per day",    agriculture_only=True) ,
    "Zambia":   feature_engineering.agricultural_wage_per_hour(df, "Zambia",      payment_frequency="Per day",    agriculture_only=False) ,
}

condition = df["farm_empl_last_12_months"] == 1

df["farm_empl_wage_per_hour"] = np.where(
    condition,
    df["country"].map(country_wage_map), 
    np.nan
)

df["farm_empl_income_per_year"] = df["farm_empl_wage_per_hour"] * df["farm_empl_hours_per_year"]

In [90]:
#SELF-EMPLOYMENT
input_costs_cols = [
    "self_empl_input_costs_last_30_days",
    "self_empl_labor_costs_last_30_days",
    "self_empl_capital_costs_last_30_days",
]
df["self_empl_input_costs"] = df[input_costs_cols].sum(axis=1, min_count=0)

df["self_empl_wage_per_month"] = df["self_empl_sales_last_30_days"] - df["self_empl_input_costs"]

df["self_empl_hours_per_month"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_days_per_week"] * 4)
    * df["self_empl_hours_per_day"],
    np.nan
)

total_safe = df["self_empl_hours_per_month"].replace(0, np.nan)
df["self_empl_wage_per_hour"] = df["self_empl_wage_per_month"] / total_safe

df["self_empl_income_per_year"] = df["self_empl_wage_per_hour"] * df["self_empl_hours_per_year"]

In [91]:
#PERMANENT WAGE EMPLOYMENT
df["wage_empl_permanent_hours_per_month"] = df["wage_empl_permanent_hours_per_year"] / 12

total_safe = df["wage_empl_permanent_hours_per_month"].replace(0, np.nan)
df["wage_empl_permanent_wage_per_hour"] = df["wage_empl_permanent_wage_per_month"] / total_safe

df["wage_empl_permanent_income_per_year"] = df["wage_empl_permanent_wage_per_hour"] * df["wage_empl_permanent_hours_per_year"]

In [92]:
#SEASONAL/CAUSAL WAGE EMPLOYMENT
df["wage_empl_seasonal_casual_wage_per_hour"] = df.apply(feature_engineering.compute_hourly_wage_for_casual_work, axis=1)

df["wage_empl_seasonal_casual_income_per_year"] = df["wage_empl_seasonal_casual_wage_per_hour"] * df["wage_empl_seasonal_casual_hours_per_year"]

In [93]:
#TOTAL YEARLY INCOME
cols = [
    "farm_empl_income_per_year",
    "self_empl_income_per_year",
    "wage_empl_permanent_income_per_year",
    "wage_empl_seasonal_casual_income_per_year",
]
df["total_income_per_year"] = df[cols].sum(axis=1, min_count=1)

### 2.5 Hourly and annual wage - relative numbers

In [94]:
#WORK TYPE SHARES
# avoid dividing by zero -> treat a total of 0 hours as NaN (undefined share)
total_safe = df["total_income_per_year"].replace(0, np.nan)

df["farm_income_annual_share"] = df["farm_empl_income_per_year"] / total_safe
df["self_empl_income_annual_share"] = df["self_empl_income_per_year"] / total_safe
df["wage_empl_permanent_income_annual_share"] = df["wage_empl_permanent_income_per_year"] / total_safe
df["wage_empl_seasonal_casual_income_annual_share"] = df["wage_empl_seasonal_casual_income_per_year"] / total_safe

In [95]:
# OFF-FARM WAGE SHARE
off_farm_cols = [
    "self_empl_income_per_year",
    "wage_empl_permanent_income_per_year",
    "wage_empl_seasonal_casual_income_per_year",
]

# sum off-farm categories, treating "not applicable" (NaN) as 0 
df["off_farm_income_per_year"] = df[off_farm_cols].sum(axis=1, min_count=0)

# but if the person has NO work data at all, keep it NaN rather than 0
df.loc[df["total_income_per_year"].isna(), "off_farm_income_per_year"] = np.nan

# share of total work time that is off-farm
total_safe = df["total_income_per_year"].replace(0, np.nan)
df["off_farm_income_annual_share"] = df["off_farm_income_per_year"] / total_safe
df["on_farm_income_annual_share"] = 1 - df["off_farm_income_annual_share"] 

### 2.5 How much more profitable is a job/own business compared to farming?

In [96]:
#This is only calculated for the hybrid workers
total_safe = df["farm_empl_wage_per_hour"].replace(0, np.nan)

df["self_empl_wage_premium_vs_farm"] = df["self_empl_wage_per_hour"] / total_safe
df["wage_empl_permanent_wage_premium_vs_farm"] = df["wage_empl_permanent_wage_per_hour"] / total_safe
df["wage_empl_seasonal_casual_wage_premium_vs_farm"] = df["wage_empl_seasonal_casual_wage_per_hour"] / total_safe

### 2.6 Main income use shares

In [97]:
employment_types    =  ["self_empl", "wage_empl_permanent", "wage_empl_seasonal_casual"]
spending_categories =  ["invest_in_own_business", "food", "education", "health", "housing_furniture", "transportation", "entertainment"]

df = feature_engineering.collapse_main_income_use(df, employment_types, spending_categories)

### 2.6 Self-employment: obstacles + financial constraints + loan source

In [98]:
#SELF-EMPLOYMENT OBSTACLES
cols = [c for c in df.columns if c.startswith('self_empl_obstacle_')]
df = feature_engineering.count_ones(df, cols, "self_empl_obstacle_index")

In [99]:
#SELF-EMPLOYMENT FINANCIAL CONSTRAINTS -> it will only give me a dummy: 1 for "yes there are constraints", 0 for "no contraints". 
cols = [c for c in df.columns if c.startswith('self_empl_three_main_finance_constr_')]
df = feature_engineering.count_ones(df, cols, "self_empl_finance_constr_index")

In [100]:
#SELF-EMPLOYMENT LOAN SOURCES
cols = [c for c in df.columns if c.startswith('self_empl_loan_source_')]
df = feature_engineering.count_ones(df, cols, "self_empl_loan_source_index")

### 2.7 Employment Keep - TODO: selecting only the ones I want to keep

In [101]:
cols_to_keep = ["personal_id",
                "total_work_hours_per_year",
                "farm_hours_annual_share",
                "self_empl_hours_annual_share",
                "wage_empl_permanent_hours_annual_share",
                "wage_empl_seasonal_casual_hours_annual_share",
                "off_farm_hours_annual_share",
                "on_farm_hours_annual_share",
                
                "farm_empl_wage_per_hour",
                "self_empl_wage_per_hour",
                "wage_empl_permanent_wage_per_hour",
                "wage_empl_seasonal_casual_wage_per_hour",
                
                "total_income_per_year",
                "farm_income_annual_share",
                "self_empl_income_annual_share",
                "wage_empl_permanent_income_annual_share",
                "wage_empl_seasonal_casual_income_annual_share",
                "off_farm_income_annual_share",
                "on_farm_income_annual_share",

                "self_empl_wage_premium_vs_farm",
                "wage_empl_permanent_wage_premium_vs_farm",
                "wage_empl_seasonal_casual_wage_premium_vs_farm",

                "main_use_invest_in_own_business_top_3",
                "main_use_food_top_3",
                "main_use_education_top_3",
                "main_use_health_top_3",
                "main_use_housing_furniture_top_3",
                "main_use_transportation_top_3",
                "main_use_entertainment_top_3",

                "self_empl_obstacle_index",
                "self_empl_finance_constr_index",
                "self_empl_loan_source_index"
                ]

In [102]:
# cols=[]
# mask = df["farm_hours_share"].notna() & df["off_farm_share"].notna()
# with pd.option_context("display.max_rows", None, "display.max_columns", None):
#     display(df.loc[mask, cols])
